#### **Importing Library**


In [41]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import time
import json
from typing import Any


from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pinecone import Pinecone

#### **Loading Environment Variables**

In [42]:
from dotenv import load_dotenv
load_dotenv(override = True)

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

#### **Loading of the PDF Data**


In [43]:
# Location of the PDF documents
folder_path = Path("data")

if not folder_path.exists():
    raise FileNotFoundError(
        f"The PDF folder was not found: {folder_path.resolve()}"
    )

pdf_files = list(folder_path.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files were found in: {folder_path.resolve()}"
    )

pdf_loader = PyPDFDirectoryLoader(str(folder_path))
raw_documents = pdf_loader.load()

empty_pages = [
    doc for doc in raw_documents
    if not doc.page_content.strip()
]


In [44]:
#checking documents after loading
print(
    f"Loaded {len(raw_documents)} pages "
    f"from {len(pdf_files)} PDF files.")

print(f"Empty or unreadable pages: {len(empty_pages)}")

if raw_documents:
    print("\nFirst document metadata:")
    print(raw_documents[0].metadata)

    print("\nFirst 500 characters:")
    print(raw_documents[0].page_content[:500])
else:
    print("No PDF documents were found.")

Loaded 21 pages from 6 PDF files.
Empty or unreadable pages: 0

First document metadata:
{'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-01-05T23:04:38+01:00', 'author': '', 'moddate': '2026-01-05T23:04:38+01:00', 'title': 'Microsoft Word - Document2', 'source': 'data\\City-guides.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}

First 500 characters:
A. New York City Guide - GrandStay ProperƟes 
Document ID: GSH-EXT-NYC-001 
Last Updated: March 20, 2024 
Applicable: All NYC properƟes 
TOP ATTRACTIONS: 
1. Must-See Landmarks: 
o Statue of Liberty & Ellis Island 
 Ferry from BaƩery Park: $24.50 adults 
 Advance Ɵckets required (book 3+ months ahead) 
 First ferry: 8:30 AM, last return: 5:00 PM 
o Empire State Building 
 Hours: 8:00 AM - 2:00 AM daily 
 Tickets: $44 standard, $77 express 
 Best Ɵmes: Early morning or aŌer 10:00 PM 
o Metr


#### **Cleaning of the text documents**


In [45]:
import re
import unicodedata


def clean_pdf_text(text: str) -> str:
    """
    Clean PDF-extracted text while preserving useful document structure
    for chunking, retrieval, and RAG.
    """

    if not isinstance(text, str):
        return ""

    # Normalize Unicode representation
    text = unicodedata.normalize("NFKC", text)

    # Normalize line endings
    text = re.sub(r"\r\n?|\f", "\n", text)

    # Replace common PDF bullets and invisible characters
    text = re.sub(r"[\uf0a7\u2022\u25cf\u25aa\u25e6\u200b\u200c\u200d\ufeff]", " ", text)

    # Remove page labels such as "Page 2" or "Page 2 of 12"
    text = re.sub(r"\bPage\s+\d+(?:\s+of\s+\d+)?\b"," ", text, flags=re.IGNORECASE)

    # Remove common repeated headers and footers
    text = re.sub(r"\bConfidential\b", " ", text, flags=re.IGNORECASE)

    # Remove decorative separators and table borders
    text = re.sub(r"[_=-]{3,}", " ", text)
    text = re.sub(r"\|+", " ", text)

    # Reduce repeated dots, while retaining a normal ellipsis
    text = re.sub(r"\.{4,}", "...", text)

    # Remove URLs and email addresses
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", " ", text)

    # Collapse spaces and tabs, but preserve newlines
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces around newlines
    text = re.sub(r" *\n *", "\n", text)

    # Keep at most one blank line between paragraphs
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    # Normalize spacing inside brackets
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)

    return text.strip()

In [46]:
#creating a new Langchain document clean_docs where only page_content changes by applying the clean_pdf_text function and metadata stays the same
clean_docs = [
    Document(
        page_content = clean_pdf_text(doc.page_content),
        metadata = doc.metadata 
    )
    for doc in raw_documents
]



In [47]:
#check clean_docs after cleaning
print(f"Raw documents: {len(raw_documents)}")
print(f"Clean documents: {len(clean_docs)}")

print("\nBefore Cleaning:\n")
print(raw_documents[0].page_content[:500])

print("\nAfter Cleaning:\n")
print(clean_docs[0].page_content[:500])

Raw documents: 21
Clean documents: 21

Before Cleaning:

A. New York City Guide - GrandStay ProperƟes 
Document ID: GSH-EXT-NYC-001 
Last Updated: March 20, 2024 
Applicable: All NYC properƟes 
TOP ATTRACTIONS: 
1. Must-See Landmarks: 
o Statue of Liberty & Ellis Island 
 Ferry from BaƩery Park: $24.50 adults 
 Advance Ɵckets required (book 3+ months ahead) 
 First ferry: 8:30 AM, last return: 5:00 PM 
o Empire State Building 
 Hours: 8:00 AM - 2:00 AM daily 
 Tickets: $44 standard, $77 express 
 Best Ɵmes: Early morning or aŌer 10:00 PM 
o Metr

After Cleaning:

A. New York City Guide - GrandStay ProperƟes
Document ID: GSH-EXT-NYC-001
Last Updated: March 20, 2024
Applicable: All NYC properƟes
TOP ATTRACTIONS:
1. Must-See Landmarks:
o Statue of Liberty & Ellis Island
Ferry from BaƩery Park: $24.50 adults
Advance Ɵckets required (book 3+ months ahead)
First ferry: 8:30 AM, last return: 5:00 PM
o Empire State Building
Hours: 8:00 AM - 2:00 AM daily
Tickets: $44 standard, $77 expres

#### **Splitting Documents into Chunks**

In [48]:
# chunking the data
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(clean_docs)
print(f"Created {len(chunks)} chunks from the PDF data")



Created 47 chunks from the PDF data


#### **Generate Embeddings and Store in Vector Pinecone Database**


In [49]:
# Initializing the Embedding Model and Pinecone
from langchain_openai import OpenAIEmbeddings
pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY"))
embedding = OpenAIEmbeddings()
test_vector = embedding.embed_query("Aurora Hospitality")

print(f"Embedding dimension: {len(test_vector)}")

Embedding dimension: 1536


In [50]:
# Creating and managing the pinecone vector store
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "aurora"
correct_dimension = 1536
cloud = "aws"
region = "us-east-1"

def wait_for_index_ready(index_name):
    while not pc.describe_index(index_name).status["ready"]:
        print("Waiting for index to be ready...")
        time.sleep(2)

def create_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=correct_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=cloud,
            region=region
        )
    )
    print(f"Created new index: {index_name}")
    wait_for_index_ready(index_name)

def add_documents_to_index(index_name, chunks, embedding):
    print(f"Adding {len(chunks)} documents/chunks to index...")

    docsearch = PineconeVectorStore.from_documents(
        documents=chunks,
        index_name=index_name,
        embedding=embedding
    )

    print(f"Successfully added {len(chunks)} documents/chunks to index.")
    return docsearch


existing_indexes = pc.list_indexes().names()

if index_name in existing_indexes:
    index_description = pc.describe_index(index_name)
    existing_dimension = index_description.dimension
    existing_metric = index_description.metric

    print(f"Index already exists: {index_name}")
    print(f"Existing index dimension: {existing_dimension}")

    if existing_dimension == correct_dimension:
        print("Index has the correct dimension.")

        index = pc.Index(index_name)
        stats = index.describe_index_stats()
        total_vector_count = stats.total_vector_count

        if total_vector_count == 0:
            print("Index exists but is empty.")
            docsearch = add_documents_to_index(index_name, chunks, embedding)
        else:
            print(f"Index already contains {total_vector_count} vectors.")
            print("Skipping document upload.")

            docsearch = PineconeVectorStore.from_existing_index(
                index_name=index_name,
                embedding=embedding
            )

    else:
        print("Index has the wrong dimension.")
        print(f"Deleting index with dimension {existing_dimension}...")

        pc.delete_index(index_name)
        time.sleep(15)

        create_index(index_name)
        docsearch = add_documents_to_index(index_name, chunks, embedding)

else:
    print(f"No index found with name: {index_name}")

    create_index(index_name)
    docsearch = add_documents_to_index(index_name, chunks, embedding)

print("Vector store is ready.")
print(f"Successfully added {len(chunks)} documents/chunks to index.")

Index already exists: aurora
Existing index dimension: 1536
Index has the correct dimension.
Index already contains 94 vectors.
Skipping document upload.
Vector store is ready.
Successfully added 47 documents/chunks to index.


In [51]:
docsearch.similarity_search("Amenities & services (wifi, breakfast, gym, pool, spa)", k=5)

[Document(id='c02f8934-04c7-4dc9-bfd3-d6e9e1922d0b', metadata={'author': '', 'creationdate': '2026-01-05T22:42:22+01:00', 'creator': 'PyPDF', 'moddate': '2026-01-05T22:42:22+01:00', 'page': 3.0, 'page_label': '4', 'producer': 'Microsoft: Print To PDF', 'source': 'data\\GSH-POL-001_Policy_manuals_v3.2_20240315.pdf', 'title': 'Microsoft Word - Document1', 'total_pages': 4.0}, page_content='in public areas o Prohibited in dining areas, pool areas,  tness centers AMENITIES PROVIDED:   Pet bed and bowls upon request   Welcome treat at check-in   Nearby walking maps   Pet-si ng services available ($30/hour)'),
 Document(id='69bc13ae-ffc7-4f74-95f7-8856380db74a', metadata={'author': '', 'creationdate': '2026-01-05T22:42:22+01:00', 'creator': 'PyPDF', 'moddate': '2026-01-05T22:42:22+01:00', 'page': 3.0, 'page_label': '4', 'producer': 'Microsoft: Print To PDF', 'source': 'data\\GSH-POL-001_Policy_manuals_v3.2_20240315.pdf', 'title': 'Microsoft Word - Document1', 'total_pages': 4.0}, page_conten

#### **Retrieval Aspect - Policy Agent Prompt**

In [52]:
#Policy Agent Prompt
from langchain.prompts import PromptTemplate

policy_prompt = PromptTemplate(
    input_variables=[
        "context",
        "question",
        "guest_type",
        "loyalty",
        "city"
    ],
    template="""
You are a POLICY INTERPRETATION AGENT for Aurora Hospitality.

ROLE
- Interpret hotel policies strictly and accurately.
- Use only the retrieved policy text.
- Do not invent, infer, or soften policy information.
- If some information is unavailable, state that clearly.
- Refer the guest to customer support only for information that cannot be confirmed.
- Do not mention retrieval systems, documents, or internal sources.

GUEST CONTEXT
Guest Type: {guest_type}
Loyalty Tier: {loyalty}
City: {city}

POLICY DOCUMENTS
{context}

USER QUESTION
{question}

CONFIDENCE SCORING
- 1.00: All parts of the answer are directly and clearly supported.
- 0.80–0.99: Most parts are directly supported; only minor details are missing.
- 0.60–0.79: The answer is partially supported, but important information is missing.
- 0.30–0.59: Only a small portion of the question is supported.
- 0.00–0.29: Little or no relevant policy information was found.

Return ONLY a valid JSON object.

Do not include

- markdown
- explanations
- comments
- code fences
- ```json

{{
    "answer": "Clear policy-based answer to the guest",
    "confidence": 0.0,
    "limitations": [],
    "policy_found":true
}}
"""
)

#### **Initialize LLM**



In [53]:
# Initializing the Language Model
from langchain.chains import ConversationalRetrievalChain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

In [54]:
# Building the policy conversational retrieval chain
policy_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=docsearch.as_retriever(),
    combine_docs_chain_kwargs={"prompt": policy_prompt},
    return_source_documents=False
)

In [55]:
chat_history = []
query = "Amenities & services (wifi, breakfast, gym, pool, spa)"

result = policy_chain.invoke(
    {"question": query, 
     "chat_history": chat_history, 
     "guest_type": "Business", 
     "loyalty": "Gold", 
     "city": "New York"})

chat_history.append((query, result["answer"]))
print(result["answer"])

{
    "answer": "The provided policies do not include information about wifi, breakfast, gym, or pool amenities. However, it does mention multiple restaurants on property, room service from 7:00 AM to 11:00 PM, and spa services available from 8:00 AM to 8:00 PM daily with advance reservations recommended and a 24-hour cancellation policy.",
    "confidence": 0.60,
    "limitations": ["No information on wifi, breakfast, gym, or pool amenities is available."],
    "policy_found": true
}


#### **Load Conversation Data**

In [56]:
#Load conversation data
conversation_file_path = 'data/gsh_support_chats_jan_jun_2024.json'

with open(conversation_file_path, 'r', encoding='utf-8') as f:
    conversation_data = json.load(f)

if not isinstance(conversation_data, list):
    raise ValueError(
        "Expected the JSON file to contain a list of conversations."
    )

print(f"{len(conversation_data)} Conversations loaded successfully.")

50 Conversations loaded successfully.


#### **Transforming Historical Conversations into LangChain Documents**

In [57]:
conversation_docs = []

for conversation in conversation_data:
    guest_message = [m['text'] for m in conversation['messages'] if m['sender'] == 'guest']
    assistant_message = [m['text'] for m in conversation['messages'] if m['sender'] == 'assistant']

    grounding_sources = []
    for msg in conversation['messages']:
        if msg.get('grounding_sources_type') and msg.get('grounding_sources_type') != 'none':
            grounding_sources.append(msg['grounding_sources_type'])
        unique_grounding_sources = list(set(grounding_sources))

    doc_content = f"""
        CONVERSATION ID: {conversation['conversation_id']},
        INTENT: {conversation['intent_primary']},
        SECONDARY INTENT: {conversation['intent_secondary'] if conversation['intent_secondary'] else 'None'},
        BRAND: {conversation['brand']},
        CITY: {conversation['property']['city']},
        COUNTRY: {conversation['property']['country']},
        GUEST TYPE: {conversation['guest_profile']['guest_type']},
        LOYALTY TIER: {conversation['guest_profile']['loyalty_tier']},
        RESOLUTION STATUS: {conversation['resolution']['status']},
        CSAT SCORE: {conversation['resolution']['csat_score']},
        TAGS: {', '.join(conversation['tags']) if conversation['tags'] else 'None'}

        KEY GUEST QUESTIONS: 
        {chr(10).join([f"- {msg}" for msg in guest_message[:5]])}
        
        SUCCESSFUL ASSISTANT RESPONSES:
        {chr(10).join([f"- {msg}" for msg in assistant_message[:5]])}

        GROUNDING SOURCES USED: {', '.join(unique_grounding_sources) if unique_grounding_sources else 'None'}

        CONVERSATION SUMMARY:
        This conversation shows how to handle {conversation['intent_primary']} inquiries for {conversation['guest_profile']['guest_type']} 
        with a {conversation['guest_profile']['loyalty_tier']} loyalty tier. The guest is from {conversation['property']['city']}, 
        {conversation['property']['country']}. The conversation was resolved with a status of {conversation['resolution']['status']} 
        and a CSAT score of {conversation['resolution']['csat_score']}.
        The assistant used sources: {', '.join(unique_grounding_sources[:2]) if unique_grounding_sources else 'general knowledge'} 
        to effectively address the guest's concerns, providing accurate information and guidance based on the hotel's policies and procedures.
 """     

    conversation_docs.append(Document(
        page_content=doc_content.strip(), 
        metadata={
            "type": "conversation",
            "conversation_id": conversation['conversation_id'],
            "intent_primary": conversation['intent_primary'],
            "secondary_intent": conversation['intent_secondary'] if conversation['intent_secondary'] else 'None',
            "brand": conversation['brand'],
            "city": conversation['property']['city'],
            "country": conversation['property']['country'],
            "guest_type": conversation['guest_profile']['guest_type'],
            "loyalty_tier": conversation['guest_profile']['loyalty_tier'],
            "resolution_status": conversation['resolution']['status'],
            "csat_score": conversation['resolution']['csat_score'],
            "tags": str(conversation['tags']),
            "source": "conversation history"
        }
            ))

print(f"Created {len(conversation_docs)} conversation documents from the JSON data.")   

            

Created 50 conversation documents from the JSON data.


#### **Clean Conversation Docs**

In [58]:
# Conversation vector store
conversation_index_name = "aurora-conversation-data"

cleaned_conversation_doc = []

for doc in conversation_docs:
    cleaned_metadata = {}
    for key, value in doc.metadata.items():
        if value is None:
            cleaned_metadata[key] = ""
        
        elif isinstance(value, (list, dict)):
            cleaned_metadata[key] = json.dumps(value)
        else:
            cleaned_metadata[key] = value
        cleaned_doc = Document(
        page_content= doc.page_content,
        metadata = cleaned_metadata
    )
    cleaned_conversation_doc.append(cleaned_doc)

print(f"cleaned {len(cleaned_conversation_doc)}")




cleaned 50


#### **Store cleaned Conversation Doc in Pinecone vector Database**

In [59]:
# storing conversational data to pinecone vector database
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import os


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

conversation_index_name = "aurora-conversation-data"
correct_dimension = 1536
cloud = "aws"
region = "us-east-1"


def wait_for_index_ready(index_name):
    while not pc.describe_index(index_name).status["ready"]:
        print("Waiting for index to be ready...")
        time.sleep(2)


def create_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=correct_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=cloud,
            region=region
        )
    )

    print(f"Created new index: {index_name}")
    wait_for_index_ready(index_name)
    
def add_documents_to_index(index_name, documents, embedding):
    if not documents:
        raise ValueError("No conversation documents were provided.")

    print(f"Adding {len(documents)} documents to index...")

    vectorstore = PineconeVectorStore.from_documents(
        documents=documents,
        index_name=index_name,
        embedding=embedding
    )

    print(f"Successfully added {len(documents)} documents to index.")
    return vectorstore


existing_indexes = pc.list_indexes().names()

if conversation_index_name in existing_indexes:
    index_description = pc.describe_index(conversation_index_name)
    existing_dimension = index_description.dimension

    print(f"Index already exists: {conversation_index_name}")
    print(f"Existing index dimension: {existing_dimension}")

    if existing_dimension == correct_dimension and index_description.metric == "cosine":
        print("Index has the correct dimension.")

        index = pc.Index(conversation_index_name)
        stats = index.describe_index_stats()
        total_vector_count = stats.total_vector_count

        if total_vector_count == 0:
            print("Index exists but is empty.")
            conversation_vectorstore = add_documents_to_index(
                conversation_index_name,
                cleaned_conversation_doc,
                embedding
            )
        else:
            print(f"Index already contains {total_vector_count} vectors.")
            print("Skipping document upload.")

            conversation_vectorstore = PineconeVectorStore.from_existing_index(
                index_name=conversation_index_name,
                embedding=embedding
            )

    else:
        print("Index has the wrong dimension.")
        print(f"Deleting index with dimension {existing_dimension}...")

        pc.delete_index(conversation_index_name)
        time.sleep(15)

        create_index(conversation_index_name)

        conversation_vectorstore = add_documents_to_index(
            conversation_index_name,
            cleaned_conversation_doc,
            embedding
        )

else:
    print(f"No index found with name: {conversation_index_name}")

    create_index(conversation_index_name)

    conversation_vectorstore = add_documents_to_index(
        conversation_index_name,
        cleaned_conversation_doc,
        embedding
    )

print("Vector store is ready.")

Index already exists: aurora-conversation-data
Existing index dimension: 1536
Index has the correct dimension.
Index already contains 50 vectors.
Skipping document upload.
Vector store is ready.


#### **Retieval Aspect for conversation agent**

In [60]:
#Defining the Conversation Intelligence Agent prompt
from langchain_core.prompts import PromptTemplate


conversation_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are the CONVERSATION INTELLIGENCE AGENT for Aurora Hospitality.

PURPOSE
Analyse how similar guest enquiries were handled in previous support
conversations and recommend an appropriate communication approach.

ROLE
- Identify useful patterns from similar historical conversations.
- Focus on tone, uncertainty handling, escalation and resolution strategy.
- Use historical conversations as behavioural guidance, not as authoritative policy.
- Do not invent hotel rules, benefits, restrictions or property details.
- If the historical context does not support part of the answer, clearly identify
  that limitation and recommend customer support only for the unsupported part.
- Do not mention historical logs, retrieval systems, databases or internal sources.
- Keep the proposed guest response polite, concise and helpful.

HISTORICAL CONVERSATIONS
{context}

USER QUESTION
{question}

CONFIDENCE GUIDANCE
- 1.00: Several closely matching successful conversations support the approach.
- 0.80-0.99: Strongly similar conversations support most of the approach.
- 0.60-0.79: Some relevant patterns exist, but important differences remain.
- 0.30-0.59: Only limited or indirectly related examples were found.
- 0.00-0.29: No useful historical pattern was found.

Return only a valid JSON object.
Do not use Markdown, code fences, comments or additional explanations.

{{
    "observed_patterns": "Summary of relevant historical handling patterns",
    "response_style": "Recommended tone and communication style",
    "conversation": "Suggested guest-facing response",
    "confidence": 0.0
}}
"""
)

print(conversation_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='\nYou are the CONVERSATION INTELLIGENCE AGENT for Aurora Hospitality.\n\nPURPOSE\nAnalyse how similar guest enquiries were handled in previous support\nconversations and recommend an appropriate communication approach.\n\nROLE\n- Identify useful patterns from similar historical conversations.\n- Focus on tone, uncertainty handling, escalation and resolution strategy.\n- Use historical conversations as behavioural guidance, not as authoritative policy.\n- Do not invent hotel rules, benefits, restrictions or property details.\n- If the historical context does not support part of the answer, clearly identify\n  that limitation and recommend customer support only for the unsupported part.\n- Do not mention historical logs, retrieval systems, databases or internal sources.\n- Keep the proposed guest response polite, concise and helpful.\n\nHISTORICAL CONVERSATIONS\n{context}\n\nUSER QUESTION\n{question}\n\

In [61]:
# Building the Conversation Intelligence Agent
from langchain.chains import RetrievalQA

conversation_retriever = conversation_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

conversation_agent = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=conversation_retriever,
    chain_type_kwargs={"prompt": conversation_prompt},
    return_source_documents=False
)


In [62]:
#Testing the Conversation Intelligence Agent
chat_history = []
query = "Amenities & services (wifi, breakfast, gym, pool, spa)"

result = conversation_agent.invoke(
    {"query": query})
conversation_output = result["result"]
chat_history.append((query, conversation_output))
print(conversation_output)

{
    "observed_patterns": "In previous conversations, inquiries about amenities such as breakfast and Wi-Fi were handled by clarifying that breakfast inclusion depends on the rate and package. The assistant provided specific breakfast hours and confirmed Wi-Fi availability, emphasizing its suitability for streaming and video calls.",
    "response_style": "The tone should be polite, concise, and informative, ensuring the guest feels acknowledged and supported. Use a friendly closing remark to enhance the guest experience.",
    "conversation": "Hello! Breakfast inclusion depends on your rate and package. Complimentary Wi-Fi is available throughout the hotel, supporting streaming and video calls. Breakfast is typically served from 6:30 to 10:30 AM on weekdays and 7:00 to 11:00 AM on weekends. If you have any other questions, feel free to ask. Safe travels!",
    "confidence": 0.8
}


#### **Converting the policy agent and conversational agent to tools**
##### *Agents without Memory (Database)*

In [63]:
# Policy interpretation agent tool
from langchain.tools import tool

#Converting the Policy RAG into a Tool
@tool
def policy_agent_tool(question: str, guest_type: str, loyalty: str, city: str) -> dict:
    """
    Tool for interpreting hotel policies based on guest information and location. It 
    retrieves relevant policy documents from the vector store and uses a language model 
    to provide accurate and detailed responses to user queries. The tool takes into account 
    the guest type, loyalty tier, and city to ensure that the information provided is applicable 
    to the specific context of the inquiry. The output reflects strict policies and procedures.
     Retrieve and interpret Aurora Hospitality policy information using
    the guest's question, guest type, loyalty tier and city.

    Returns a structured dictionary containing:
    - answer
    - confidence
    - limitations
    """
    # Implementation for policy interpretation
    chain_result = policy_chain.invoke(
        {
            "question": question,
            "chat_history": chat_history,
            "guest_type": guest_type,
            "loyalty": loyalty,
            "city": city
        }
    )

    # ConversationalRetrievalChain stores the generated output under "answer"
    raw_policy_output = chain_result.get("answer", "")

    # Convert the JSON string into a Python dictionary
    policy_output = parse_agent_output(raw_policy_output)

    # Guarantee that all expected fields exist
    return {
        "answer": policy_output.get(
            "answer",
            "No confirmed policy information was found."
        ),
        "confidence": float(
            policy_output.get("confidence", 0.5)
        ),
        "limitations": policy_output.get("limitations", [])
    }

In [64]:
# Conversational analyzer agent tool

@tool
def conversation_agent_tool(question: str) -> dict:
    """
    Analyse historical Aurora Hospitality conversations to identify
    relevant handling patterns, response styles, escalation approaches
    and resolution strategies for the current guest enquiry.

    Returns a structured dictionary containing:
    - observed_patterns
    - response_style
    - conversation
    - confidence
    """

    chain_result = conversation_agent.invoke({"query": question})

    # RetrievalQA stores the generated response under "result"
    raw_conversation_output = chain_result.get("result", "")

    # Convert the JSON-formatted string to a Python dictionary
    conversation_output = parse_agent_output(raw_conversation_output)

    return {
        "observed_patterns": conversation_output.get(
            "observed_patterns",
            "No closely matching historical handling pattern was identified."
        ),
        "response_style": conversation_output.get(
            "response_style",
            "Polite, clear and concise."
        ),
        "conversation": conversation_output.get(
            "conversation",
            "The available historical conversations do not provide enough "
            "information to recommend a complete response."
        ),
        "confidence": float(
            conversation_output.get("confidence", 0.5)
        )
    }

#### **Initialize Database** 

In [65]:
# Create a database that serves as agents memory
import sqlite3
db = sqlite3.connect('conversation_memory.db', check_same_thread=False)

db.execute(""" 
    CREATE TABLE IF NOT EXISTS conversation_memory(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id TEXT,
        role TEXT,
        message TEXT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

# Create an index on session_id
db.execute("""
CREATE INDEX IF NOT EXISTS idx_session
ON conversation_memory(session_id)
""")

db.commit()

#### **Storing Conversation History**

In [66]:
# Populate the database
def store_conversation(session_id: str, role: str, message: str):
      # Validate role
    if role not in ("user", "assistant"):
        raise ValueError("Role must be either 'user' or 'assistant'.")

    # Prevent empty messages
    if not message or not message.strip():
        print("Empty message. Nothing was stored.")
        return

    try:
        """
        Store a conversation message in the database with the session ID, role (user or assistant), 
        message content, and timestamp. This function allows for tracking and analyzing conversations 
        over time, providing a historical record of interactions for future reference and analysis.
        """
        db.execute(
            "INSERT INTO conversation_memory (session_id, role, message) VALUES (?, ?, ?)",
            (session_id, role, message)
        )
        db.commit()
    except Exception as e:
        print(f"Database error: {e}")

#### **Retrieving Conversation History**

In [67]:
# Getting each conversation one by one
def get_conversations(session_id: str, limit=6):

    """
    Retrieve a list of conversation messages from the database for a specific session ID, 
    limited to the most recent 'limit' number of messages. This function allows for fetching 
    historical conversation data for analysis or display, providing context for ongoing interactions.
    """ 
    if not rows:
        return "No previous conversation history."
    try:
        rows = db.execute(
            """SELECT role, message, timestamp FROM conversation_memory WHERE session_id = ? ORDER BY 
            timestamp ASC LIMIT ?""",
            (session_id, limit)
        ).fetchall()
       
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    
    return "\n".join(f"{role}: {msg}, {timestamp}" for role, message, timestamp in reversed(rows))

#### **Retrieving Conversation History as User–Assistant Pairs**

In [68]:
# Retrieve and keep track of user messages and assistant responses
def get_chat_history_tuples(session_id: str, limit: int = 6):
    """
    Return the most recent user/assistant message pairs for a session.

    The messages are returned in chronological order as:
    [
        (user_message, assistant_response),
        ...
    ]
    """
    try:
        rows = db.execute(
            """
            SELECT role, message, timestamp
            FROM (
                SELECT role, message, timestamp
                FROM conversation_memory
                WHERE session_id = ?
                ORDER BY timestamp DESC
                LIMIT ?
            )
            ORDER BY timestamp ASC
            """,
            (session_id, limit * 2)
        ).fetchall()
    
    
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        return []

    history = []
    current_user_msg = None

    for role, message, timestamp in rows:
        if role == "user":
            current_user_msg = message

        elif role == "assistant" and current_user_msg is not None:
            history.append((current_user_msg, message))
            current_user_msg = None

    return history[-limit:]

#### **Retrieving Conversation History as Formatted Text**

In [69]:
def get_chat_history_text(session_id: str, limit: int = 6) -> str:
    """
    Return the most recent conversation messages as formatted text.
    """

    rows = db.execute(
        """
        SELECT role, message
        FROM (
            SELECT role, message, timestamp
            FROM conversation_memory
            WHERE session_id = ?
            ORDER BY timestamp DESC
            LIMIT ?
        )
        ORDER BY timestamp ASC
        """,
        (session_id, limit * 2)
    ).fetchall()

    return "\n".join(
        f"{role}: {message}"
        for role, message in rows
    )

#### **Agents with Memory (database)**
##### **Updating the Policy Agent Tool with Conversation Memory**

In [70]:
@tool
def policy_agent_tool(
    question: str,
    guest_type: str,
    loyalty: str,
    city: str,
    chat_history: list
) -> dict[str, Any]:
    """
    Retrieve and interpret official Aurora Hospitality policy information
    using the current question, guest profile, city and recent conversation
    history.
    """

    if not question or not question.strip():
        return {
            "answer": "No question was provided.",
            "confidence": 0.0,
            "limitations": "The Policy Agent requires a valid question."
        }

    result = policy_chain.invoke(
        {
            "question": question.strip(),
            "chat_history": chat_history,
            "guest_type": guest_type,
            "loyalty": loyalty,
            "city": city
        }
    )

    raw_output = result.get("answer", "")

    try:
        parsed_output = json.loads(raw_output)

        return {
            "answer": parsed_output.get("answer", ""),
            "confidence": float(parsed_output.get("confidence", 0.5)),
            "limitations": parsed_output.get("limitations", "")
        }

    except (json.JSONDecodeError, TypeError, ValueError):
        return {
            "answer": raw_output,
            "confidence": 0.5,
            "limitations": "The response could not be parsed as structured JSON."
        }

#### **Updating the Conversation Agent Tool with Conversation Memory**

In [71]:
@tool
def conversation_agent_tool(
    conversation_query: str,
    chat_history_text: str,
    guest_type: str,
    loyalty: str,
    city: str
):
    """
    Analyze historical support conversations and recommend how a similar
    guest enquiry should be handled.
    """

    if not conversation_query or not conversation_query.strip():
        return {
            "observed_patterns": "",
            "response_style": "",
            "conversation": "No valid guest question was provided.",
            "confidence": 0.0
        }

    combined_query = f"""

    GUEST PROFILE:
    Guest Type: {guest_type}
    Loyalty Tier: {loyalty}
    City: {city}

    RECENT CONVERSATION WITH THE CURRENT GUEST:
    {chat_history_text or "No previous conversation is available."}

    CURRENT GUEST QUESTION:
    {conversation_query}
    """

    result = conversation_agent.invoke({
        "query": combined_query
    })

    raw_output = result.get("result", "")

    try:
        parsed_output = json.loads(raw_output)

        return {
            "observed_patterns": parsed_output.get("observed_patterns", ""),
            "response_style": parsed_output.get("response_style", ""),
            "conversation": parsed_output.get("conversation", ""),
            "confidence": float(parsed_output.get("confidence", 0.5))
        }

    except (json.JSONDecodeError, TypeError, ValueError):
        return {
            "observed_patterns": "",
            "response_style": "",
            "conversation": raw_output,
            "confidence": 0.5
        }

#### **Creating the Aggregator or Synthethizer Agent**

In [72]:
#Final Response Aggregator Prompt
from langchain_core.prompts import PromptTemplate


aggregator_prompt = PromptTemplate(
    input_variables=[
        "policy_agent_output",
        "conversation_agent_output",
        "chat_history_text",
        "question"
    ],

    template="""
You are the FINAL RESPONSE AGGREGATOR for Aurora Hospitality.

Your task is to combine the authoritative policy interpretation with
conversation-based communication guidance and produce one final response
to the guest in English.

DECISION RULES

1. Treat the Policy Agent output as the authoritative source for hotel
   rules, eligibility, restrictions and permitted actions.

2. Use the Conversation Agent output only to guide tone, empathy,
   explanation style, escalation and practical handling.

3. If the outputs conflict, follow the Policy Agent output. The
   Conversation Agent may still guide how the answer is communicated.

4. Use the confidence values internally to assess uncertainty, but never
   reveal confidence scores to the guest.

5. If no relevant conversation pattern is available, answer using the
   Policy Agent output.

6. If the policy information is incomplete or unclear, answer only the
   parts supported by the available information, include a brief
   disclaimer and recommend contacting Aurora Hospitality customer
   support for confirmation.

7. If no relevant policy is available, do not treat historical
   conversations as official policy. Provide only safe general guidance
   and recommend customer support.

INPUTS

POLICY AGENT OUTPUT:
{policy_agent_output}

CONVERSATION AGENT OUTPUT:
{conversation_agent_output}

ORIGINAL USER QUESTION:
{question}

TASK

Synthesize the inputs into one customer-facing response that:

- directly answers the original question;
- follows the official policy information;
- is clear, practical and actionable;
- sounds professional, natural and empathetic;
- uses relevant conversation guidance without presenting it as policy;
- includes a brief disclaimer only when necessary.

FINAL ANSWER REQUIREMENTS

- Write between 3 and 5 sentences.
- Use clear, natural and customer-friendly English.
- Do not mention agents, retrieval systems, documents, internal sources
  or confidence scores.
- Do not expose internal reasoning.
- Refer the guest to customer support only when clarification or further
  action is genuinely required.

FINAL ANSWER:
"""
)

#### **Creating the Aggregator Chain**

In [73]:
from langchain.chains import LLMChain

aggregator_chain = LLMChain(
    llm=llm,
    prompt = aggregator_prompt
)

In [74]:
import asyncio
import json


def normalise_tool_result(result):
    """
    Convert a tool result into a Python dictionary where possible.

    Tool outputs may be dictionaries, JSON-formatted strings,
    or wrapper dictionaries containing an 'output' or 'result' field.
    """

    if isinstance(result, dict):
        # Some tool wrappers place the actual result inside
        # an 'output' field.
        if (
            "output" in result
            and len(result) == 1
        ):
            result = result["output"]

        # Only unwrap 'result' when it contains the complete
        # structured output rather than one field of it.
        elif (
            "result" in result
            and len(result) == 1
        ):
            result = result["result"]

        else:
            return result

    if isinstance(result, str):
        try:
            parsed_result = json.loads(result)

            if isinstance(parsed_result, dict):
                return parsed_result

        except json.JSONDecodeError:
            return {
                "raw_output": result
            }

    return {
        "raw_output": str(result)
    }


async def run_agents_parallel(
    question: str,
    guest_type: str,
    loyalty: str,
    city: str,
    chat_history_tuples,
    chat_history_text: str
):
    """
    Run the Policy Agent and Conversation Agent concurrently.

    The Policy Agent Tool receives:
    - question
    - guest_type
    - loyalty
    - city
    - chat_history

    The Conversation Agent Tool receives:
    - conversation_query
    - guest_type
    - loyalty
    - city
    - chat_history_text
    """

    # Input expected by the Policy Agent Tool
    policy_input = {
        "question": question,
        "guest_type": guest_type,
        "loyalty": loyalty,
        "city": city,
        "chat_history": chat_history_tuples
    }

    # Input expected by the Conversation Agent Tool
    conversation_input = {
        "conversation_query": question,
        "guest_type": guest_type,
        "loyalty": loyalty,
        "city": city,
        "chat_history_text": (
            chat_history_text
            or "No previous conversation history."
        )
    }

    # Run both synchronous LangChain tools in worker threads
    # so they can execute concurrently.
    policy_task = asyncio.to_thread(
        policy_agent_tool.invoke,
        policy_input
    )

    conversation_task = asyncio.to_thread(
        conversation_agent_tool.invoke,
        conversation_input
    )

    policy_result, conversation_result = (
        await asyncio.gather(
            policy_task,
            conversation_task
        )
    )

    # Preserve the complete structured outputs, including
    # confidence scores and limitations.
    policy_output = normalise_tool_result(
        policy_result
    )

    conversation_output = normalise_tool_result(
        conversation_result
    )

    return policy_output, conversation_output

In [75]:
async def agentic_rag_answer(
    question: str,
    guest_type: str,
    loyalty: str,
    city: str,
    session_id: str
):
    chat_history_tuples = get_chat_history_tuples(
        session_id=session_id,
        limit=6
    )

    chat_history_text = get_chat_history_text(
        session_id=session_id,
        limit=6
    )

    policy_output, conversation_output = await run_agents_parallel(
        question=question,
        guest_type=guest_type,
        loyalty=loyalty,
        city=city,
        chat_history_tuples=chat_history_tuples,
        chat_history_text=chat_history_text
    )

    final_answer = aggregator_chain.invoke(
        {
            "policy_agent_output": json.dumps(policy_output, indent=2),
            "conversation_agent_output": json.dumps(conversation_output, indent=2),
            "chat_history_text": chat_history_text,
            "question": question
        }
    )

    if isinstance(final_answer, dict):
        answer_text = (
            final_answer.get("text")
            or final_answer.get("answer")
            or final_answer.get("result")
        )
    else:
        answer_text = str(final_answer)

    if not answer_text:
        raise ValueError(
            f"No recognised answer field returned by aggregator_chain: "
            f"{final_answer}"
        )

    store_conversation(session_id, "user", question)
    store_conversation(session_id, "assistant", answer_text)

    return {
        "text": answer_text,
        "policy_output": policy_output,
        "conversation_output": conversation_output
    }

In [76]:
session_id = "guest_001"

response = await agentic_rag_answer(
    question="Amenities & services (wifi, breakfast, gym, pool, spa)",
    guest_type="Business",
    loyalty="Gold",
    city="Sydney",
    session_id=session_id
)

print(response["text"])


Thank you for your inquiry about our amenities and services! We offer multiple restaurants on property, poolside service from 11:00 AM to 6:00 PM, room service from 7:00 AM to 11:00 PM, and spa services daily from 8:00 AM to 8:00 PM, with advance reservations recommended and a 24-hour cancellation policy. While I don't have specific details about Wi-Fi, gym, and pool services at this time, I recommend reaching out to our customer support team for the most accurate information. If you have any further questions, feel free to ask!


In [77]:
session_id = "guest_001"

response = await agentic_rag_answer(
    question="Anyways, what time is breakfast going to be served",
    guest_type="Business",
    loyalty="Gold",
    city="Sydney",
    session_id=session_id
)

print(response["text"])

Thank you for your inquiry about breakfast times! Breakfast is served from 6:30 AM to 10:30 AM on weekdays and from 7:00 AM to 11:00 AM on weekends. Please keep in mind that specific details may vary by location. If you have any further questions or need additional information, feel free to ask!


In [78]:
session_id = "guest_001"

response = await agentic_rag_answer(
    question="What about Lunch?",
    guest_type="Business",
    loyalty="Gold",
    city="Sydney",
    session_id=session_id
)

print(response["text"])

Thank you for your inquiry about lunch! Lunch is served in the Main Restaurant from 11:30 AM to 2:30 PM. If you have any further questions or need additional assistance, please feel free to reach out. We're here to help and ensure you have a wonderful dining experience!


In [79]:
session_id = "guest_001"

response = await agentic_rag_answer(
    question="And dinner?",
    guest_type="Business",
    loyalty="Gold",
    city="Sydney",
    session_id=session_id
)

print(response["text"])

Thank you for your inquiry about dinner! Dinner is served at the Main Restaurant from 5:30 PM to 10:00 PM. If you have any further questions or need assistance, please feel free to reach out. We're here to help and ensure you have a wonderful dining experience!


The primary difference is that ConversationalRetrievalChain dynamically rewrites the questions using chat history before searching the database, whereas RetrievalQA searches the database using the exact, raw query. RetrievalQA handles isolated, single-turn questions. ConversationalRetrievalChain handles ongoing, multi-turn conversations.

Behind the Scenes: 

RetrievalQA: RetrievalQA runs a straightforward, single-step retrieval pipeline. It assumes every question stands completely on its own.

[ User Query ] ──► [ Vector Store Search ] ──► [ LLM + Context ] ──► [ Final Answer ]

The Vector Search: When a query is passed (e.g., "How much did we spend on it?"). RetrievalQA converts this exact string into an embedding vector. It immediately queries the vector database.

The Retrieval Flaw: If the query contains pronouns like "it," "they," or "he," the vector database performs poorly. It searches for the literal word "it" instead of the actual subject discussed in previous turns.

The Generation: The system passes whatever chunks it finds to the LLM alongside the raw question. The LLM generates the final response.

ConversationalRetrievalChain:

ConversationalRetrievalChain runs a two-step LLM process. It uses a specialized sub-chain to resolve pronouns and context before searching the database.                        
                        ┌──────────────────┐
                        │   Chat History   │
                        └────────┬─────────┘
                                 ▼
[ New User Query ] ──► [ LLM: Question Re-writer ] ──► [ Standalone Query ]
                                                               │
                                                               ▼
[ Final Answer ] ◄── [ LLM + Context ] ◄── [ Vector Store ] ◄──┘

Step 1: The Context condensation (The "Secret Sauce")When a new question is submitted, the chain does not search the database right away. Instead, it bundles two things together and sends them to the LLM:
- The entire Chat History (past questions and answers)
- Your New Follow-Up Question

It instructs the LLM: "Read this history and rewrite the new question so it can be understood completely on its own, without needing the history.

"Example: If the history is talking about "Project Alpha", and the new question is "How much did we spend on it?", the re-writer LLM outputs a brand-new string: "How much money was spent on Project Alpha?"

Step 2: The Standard RAG Pipeline 

The chain takes that newly generated, standalone question and passes it into a standard retrieval flow:
- It embeds the rewritten question and searches the vector database.
- The database returns highly accurate, relevant text chunks because the vague pronouns were resolved.
- The chain sends these precise chunks, along with the original question and chat history, to the final LLM to generate a natural, context-aware reply.

Summary of LLM Calls

RetrievalQA: Uses exactly one LLM call per user interaction (to answer the question based on the retrieved documents).

ConversationalRetrievalChain: Uses two LLM calls for every follow-up interaction. Call one rewrites the question; call two generates the final answer.

#### **Evaluation**

In [80]:
# Evaluation metrics


# Calculate Recall@k
def compute_recall_at_k(retrieved_docs, relevant_docs, k=5):
    """
    Calculate the proportion of relevant documents found
    within the top-k retrieved documents.
    """

    if not relevant_docs:
        return 0.0

    retrieved_at_k = retrieved_docs[:k]

    relevant_retrieved = sum(
        document in relevant_docs
        for document in retrieved_at_k
    )

    return relevant_retrieved / len(relevant_docs)


# Calculate mean reciprocal rank for one query
def compute_mean_reciprocal_rank(retrieved_docs, relevant_docs):
    """
    Calculate the reciprocal rank of the first relevant document.
    """

    for i, document in enumerate(retrieved_docs, start=1):
        if document in relevant_docs:
            return 1.0 / i

    return 0.0


# Evaluate faithfulness
def evaluate_faithfulness(response, context):
    """
    Evaluation metric that uses an LLM to assess whether
    the generated response is supported by the retrieved context.
    """

    prompt = f"""
Rate how faithful the response is to the provided context.

Context:
{context}

Response:
{response}

Return only one number between 0 and 1.

1 = Completely supported by the context
0 = Not supported by the context

Score:
""".strip()

    try:
        result = llm.invoke(prompt)
        score = float(result.content.strip())

        return min(max(score, 0.0), 1.0)

    except (ValueError, TypeError, AttributeError) as error:
        print(f"Faithfulness evaluation error: {error}")
        return 0.5


# Evaluate relevance
def evaluate_relevance(response, question):
    """
    Evaluation metric that uses an LLM to assess whether
    the generated response answers the user's question.
    """

    prompt = f"""
Rate how relevant the response is to the provided question.

Question:
{question}

Response:
{response}

Return only one number between 0 and 1.

1 = Completely relevant
0 = Not relevant

Score:
""".strip()

    try:
        result = llm.invoke(prompt)
        score = float(result.content.strip())

        return min(max(score, 0.0), 1.0)

    except (ValueError, TypeError, AttributeError) as error:
        print(f"Relevance evaluation error: {error}")
        return 0.5

#### **Caching and Cost Tracking Mechanism**

In [81]:
# PERSISTENT CACHE, CACHE METRICS AND COST TRACKING

import hashlib
import json
import sqlite3
from datetime import datetime, timezone


# CONFIGURATION
CACHE_DB_PATH = "aurora_cache.db"
MAX_CACHE_ENTRIES = 100

CONFIDENCE_THRESHOLD = 0.70

# Stores evaluation results
retrieval_results = []
generation_results = []

# Stores token usage and estimated cost information
cost_log = []
token_usage_log = []

# Configurable estimated GPT-4o-mini costs per 1,000 tokens
INPUT_COST_PER_1K = 0.00015
OUTPUT_COST_PER_1K = 0.00060

# Cache counters for the current application run
cache_metrics = {
    "hits": 0,
    "misses": 0
}


# HELPER FUNCTIONS
def normalise_cache_value(value):
    """
    Convert a cache-key value into a consistent string format.

    This prevents errors when a value is None and ensures that
    differences in capitalisation or surrounding whitespace do
    not create unnecessary cache entries.
    """

    if value is None:
        return ""

    return str(value).strip().lower()


def get_utc_timestamp():
    """
    Return the current UTC timestamp in ISO 8601 format.
    """

    return datetime.now(timezone.utc).isoformat()



# INITIALISE THE PERSISTENT CACHE DATABASE
def initialise_cache_database():
    """
    Create the SQLite table used for persistent response caching.
    """

    with sqlite3.connect(CACHE_DB_PATH) as connection:
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS response_cache (
                cache_key TEXT PRIMARY KEY,
                question TEXT NOT NULL,
                guest_type TEXT,
                loyalty TEXT,
                city TEXT,
                session_id TEXT,
                response_json TEXT NOT NULL,
                created_at TEXT NOT NULL,
                last_accessed_at TEXT NOT NULL,
                access_count INTEGER NOT NULL DEFAULT 1
            )
            """
        )

        connection.commit()


# GENERATE A STABLE CACHE KEY
def get_cache_key(
    question,
    guest_type,
    loyalty,
    city,
    session_id
):
    """
    Create a stable SHA-256 cache key from the user's question,
    profile information and session ID.

    Conversation history is deliberately excluded so that the
    key remains stable as conversation memory is updated.
    """

    cache_input = {
        "question": normalise_cache_value(question),
        "guest_type": normalise_cache_value(guest_type),
        "loyalty": normalise_cache_value(loyalty),
        "city": normalise_cache_value(city),
        "session_id": normalise_cache_value(session_id)
    }

    cache_string = json.dumps(
        cache_input,
        sort_keys=True,
        ensure_ascii=False
    )

    return hashlib.sha256(
        cache_string.encode("utf-8")
    ).hexdigest()



# DETERMINE WHETHER A QUESTION SHOULD BE CACHED
def is_cacheable_question(question):
    """
    Determine whether a question is suitable for caching.

    Clear standalone questions may use the cache. Short or
    context-dependent follow-up questions bypass the cache.
    """

    normalised_question = normalise_cache_value(question)

    if not normalised_question:
        return False

    follow_up_phrases = [
        "what about",
        "how about",
        "and what about",
        "and how about",
        "what time does it",
        "does it",
        "is it",
        "can it",
        "can they",
        "what about that",
        "how about that",
        "that one",
        "this one",
        "the same one"
    ]

    return not any(
        normalised_question.startswith(phrase)
        for phrase in follow_up_phrases
    )


# RETRIEVE A RESPONSE FROM THE CACHE
def get_cached_response(
    question,
    guest_type,
    loyalty,
    city,
    session_id
):
    """
    Retrieve a response that exactly matches the generated
    cache key.

    On a cache hit, update the access count and most recent
    access time.
    """

    cache_key = get_cache_key(
        question=question,
        guest_type=guest_type,
        loyalty=loyalty,
        city=city,
        session_id=session_id
    )

    with sqlite3.connect(CACHE_DB_PATH) as connection:
        cursor = connection.cursor()

        cursor.execute(
            """
            SELECT response_json
            FROM response_cache
            WHERE cache_key = ?
            """,
            (cache_key,)
        )

        row = cursor.fetchone()

        if row is None:
            cache_metrics["misses"] += 1
            print("Cache miss: no matching response found.")
            return None

        cursor.execute(
            """
            UPDATE response_cache
            SET last_accessed_at = ?,
                access_count = access_count + 1
            WHERE cache_key = ?
            """,
            (
                get_utc_timestamp(),
                cache_key
            )
        )

        connection.commit()

    try:
        cached_response = json.loads(row[0])
    except json.JSONDecodeError as error:
        raise ValueError(
            "The cached response contains invalid JSON."
        ) from error

    if not isinstance(cached_response, dict):
        cached_response = {
            "text": str(cached_response)
        }

    cached_response["cache_hit"] = True

    cache_metrics["hits"] += 1

    print(
        "Cache hit: returning the previously stored response."
    )

    return cached_response


# STORE A RESPONSE IN THE CACHE
def cache_response(
    question,
    guest_type,
    loyalty,
    city,
    response,
    session_id
):
    """
    Store a generated response in the persistent cache.

    If the cache key already exists, update the stored response
    and access time without resetting the original creation time.
    """

    cache_key = get_cache_key(
        question=question,
        guest_type=guest_type,
        loyalty=loyalty,
        city=city,
        session_id=session_id
    )

    current_time = get_utc_timestamp()

    response_json = json.dumps(
        response,
        ensure_ascii=False,
        default=str
    )

    with sqlite3.connect(CACHE_DB_PATH) as connection:
        connection.execute(
            """
            INSERT INTO response_cache (
                cache_key,
                question,
                guest_type,
                loyalty,
                city,
                session_id,
                response_json,
                created_at,
                last_accessed_at,
                access_count
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)

            ON CONFLICT(cache_key) DO UPDATE SET
                question = excluded.question,
                guest_type = excluded.guest_type,
                loyalty = excluded.loyalty,
                city = excluded.city,
                session_id = excluded.session_id,
                response_json = excluded.response_json,
                last_accessed_at = excluded.last_accessed_at
            """,
            (
                cache_key,
                str(question),
                str(guest_type),
                str(loyalty),
                str(city),
                str(session_id),
                response_json,
                current_time,
                current_time,
                1
            )
        )

        connection.commit()

    enforce_cache_limit()



# LIMIT THE NUMBER OF STORED CACHE ENTRIES
def enforce_cache_limit(
    max_entries=MAX_CACHE_ENTRIES
):
    """
    Remove the least recently accessed entries when the cache
    exceeds its configured maximum size.
    """

    if max_entries < 1:
        raise ValueError(
            "max_entries must be at least 1."
        )

    with sqlite3.connect(CACHE_DB_PATH) as connection:
        cursor = connection.execute(
            """
            SELECT COUNT(*)
            FROM response_cache
            """
        )

        cache_size = cursor.fetchone()[0]
        number_to_delete = cache_size - max_entries

        if number_to_delete > 0:
            connection.execute(
                """
                DELETE FROM response_cache
                WHERE cache_key IN (
                    SELECT cache_key
                    FROM response_cache
                    ORDER BY last_accessed_at ASC
                    LIMIT ?
                )
                """,
                (number_to_delete,)
            )

            connection.commit()


# RETURN AND DISPLAY CACHE PERFORMANCE
def get_cache_summary(display=True):
    """
    Calculate and return cache-performance metrics.

    When display is True, also print the metrics.
    """

    hits = cache_metrics["hits"]
    misses = cache_metrics["misses"]
    total_requests = hits + misses

    hit_rate = (
        hits / total_requests
        if total_requests > 0
        else 0.0
    )

    with sqlite3.connect(CACHE_DB_PATH) as connection:
        cursor = connection.execute(
            """
            SELECT
                COUNT(*),
                COALESCE(SUM(access_count), 0),
                COALESCE(MAX(last_accessed_at), '')
            FROM response_cache
            """
        )

        stored_entries, total_accesses, last_accessed = (
            cursor.fetchone()
        )

    summary = {
        "hits": hits,
        "misses": misses,
        "total_requests": total_requests,
        "hit_rate": hit_rate,
        "stored_entries": stored_entries,
        "total_cache_accesses": total_accesses,
        "last_accessed_at": last_accessed or None
    }

    if display:
        print("Cache summary")
        print(f"Cache hits: {summary['hits']}")
        print(f"Cache misses: {summary['misses']}")
        print(
            f"Total cache checks: "
            f"{summary['total_requests']}"
        )
        print(
            f"Cache hit rate: "
            f"{summary['hit_rate']:.2%}"
        )
        print(
            f"Persistent cache entries: "
            f"{summary['stored_entries']}"
        )

    return summary

# CLEAR THE PERSISTENT CACHE
def clear_response_cache():
    """
    Delete all responses from the persistent cache and reset
    the in-memory cache counters.
    """

    with sqlite3.connect(CACHE_DB_PATH) as connection:
        connection.execute(
            """
            DELETE FROM response_cache
            """
        )

        connection.commit()

    cache_metrics["hits"] = 0
    cache_metrics["misses"] = 0

    print("Persistent response cache cleared.")


# TRACK TOKEN USAGE AND ESTIMATED COST
def track_token_usage(
    agent_name,
    input_tokens,
    output_tokens,
    latency_ms=0
):
    """
    Calculate and record token usage and estimated model cost
    for one LLM call.
    """

    input_tokens = int(input_tokens)
    output_tokens = int(output_tokens)
    latency_ms = float(latency_ms)

    if input_tokens < 0 or output_tokens < 0:
        raise ValueError(
            "Token counts cannot be negative."
        )

    if latency_ms < 0:
        raise ValueError(
            "Latency cannot be negative."
        )

    input_cost = (
        input_tokens / 1000
    ) * INPUT_COST_PER_1K

    output_cost = (
        output_tokens / 1000
    ) * OUTPUT_COST_PER_1K

    total_cost = input_cost + output_cost

    usage_entry = {
        "agent": str(agent_name),
        "timestamp": get_utc_timestamp(),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
        "latency_ms": latency_ms
    }

    token_usage_log.append(usage_entry)
    cost_log.append(total_cost)

    return total_cost



# DISPLAY COST SUMMARY
def get_cost_summary():
    """
    Calculate, display and return accumulated token usage and
    estimated model-call cost.
    """

    if not token_usage_log:
        print("No token usage has been tracked yet.")

        return {
            "total_calls": 0,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "total_cost": 0.0,
            "average_cost_per_call": 0.0,
            "average_latency_ms": 0.0
        }

    total_input_tokens = sum(
        entry["input_tokens"]
        for entry in token_usage_log
    )

    total_output_tokens = sum(
        entry["output_tokens"]
        for entry in token_usage_log
    )

    total_tokens = (
        total_input_tokens
        + total_output_tokens
    )

    total_cost = sum(
        entry["total_cost"]
        for entry in token_usage_log
    )

    total_latency = sum(
        entry["latency_ms"]
        for entry in token_usage_log
    )

    total_calls = len(token_usage_log)

    summary = {
        "total_calls": total_calls,
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "total_tokens": total_tokens,
        "total_cost": total_cost,
        "average_cost_per_call": (
            total_cost / total_calls
        ),
        "average_latency_ms": (
            total_latency / total_calls
        )
    }

    print("Cost summary")
    print(
        f"Total model calls tracked: "
        f"{summary['total_calls']}"
    )
    print(
        f"Total input tokens: "
        f"{summary['input_tokens']}"
    )
    print(
        f"Total output tokens: "
        f"{summary['output_tokens']}"
    )
    print(
        f"Total tokens used: "
        f"{summary['total_tokens']}"
    )
    print(
        f"Total estimated cost: "
        f"${summary['total_cost']:.6f}"
    )
    print(
        f"Average estimated cost per call: "
        f"${summary['average_cost_per_call']:.6f}"
    )
    print(
        f"Average tracked model latency: "
        f"{summary['average_latency_ms']:.0f} ms"
    )

    return summary


# Create the cache database and table when this cell runs
initialise_cache_database()

print("Persistent cache database initialised successfully.")

Persistent cache database initialised successfully.


In [82]:
# COMBINED AGENT-CONFIDENCE CALCULATION

import json


def normalise_confidence(value, default=0.5):
    """
    Convert a confidence value to a float and constrain it
    to the valid range of 0.0 to 1.0.

    If conversion fails, return the supplied default value.
    """

    try:
        confidence = float(value)

    except (TypeError, ValueError):
        confidence = float(default)

    return min(max(confidence, 0.0), 1.0)


def parse_agent_output(output):
    """
    Convert an agent output into a Python dictionary.

    The output may already be a dictionary or may be returned
    as a JSON-formatted string.
    """

    if isinstance(output, dict):
        return output

    if isinstance(output, str):
        try:
            parsed_output = json.loads(output)

        except json.JSONDecodeError:
            return {}

        if isinstance(parsed_output, dict):
            return parsed_output

    return {}


def normalise_limitations(limitations):
    """
    Convert the Policy Agent's limitations into a clean list.

    Empty values and statements such as 'none' or
    'no limitations' are removed so that they do not
    incorrectly trigger a confidence penalty.
    """

    if limitations is None:
        return []

    if isinstance(limitations, str):
        limitations = [limitations]

    elif not isinstance(limitations, list):
        limitations = [str(limitations)]

    empty_limitation_values = {
        "",
        "none",
        "no limitation",
        "no limitations",
        "not applicable",
        "n/a"
    }

    cleaned_limitations = []

    for limitation in limitations:
        limitation_text = str(limitation).strip()

        if limitation_text.lower() not in empty_limitation_values:
            cleaned_limitations.append(limitation_text)

    return cleaned_limitations


def calculate_confidence_score(
    policy_output,
    conversation_output
):
    """
    Calculate a combined confidence score from the Policy Agent
    and Conversation Agent outputs.

    The Policy Agent receives 60% of the total weight because
    official policy documents are more authoritative than
    historical conversation patterns.

    The Conversation Agent receives 40% of the total weight.

    A 20% reduction is applied when the Policy Agent reports
    meaningful limitations.
    """

    policy_data = parse_agent_output(policy_output)
    conversation_data = parse_agent_output(
        conversation_output
    )

    policy_confidence = normalise_confidence(
        policy_data.get("confidence"),
        default=0.5
    )

    conversation_confidence = normalise_confidence(
        conversation_data.get("confidence"),
        default=0.5
    )

    limitations = normalise_limitations(
        policy_data.get("limitations")
    )

    # Give official policy knowledge greater weight.
    base_confidence = (
        policy_confidence * 0.60
        + conversation_confidence * 0.40
    )

    # Apply a 20% penalty when meaningful policy
    # limitations have been reported.
    limitation_penalty_applied = bool(limitations)

    if limitation_penalty_applied:
        base_confidence *= 0.80

    final_confidence = round(
        min(max(base_confidence, 0.0), 1.0),
        2
    )

    print("Confidence calculation")
    print(f"Policy confidence: {policy_confidence:.2f}")
    print(
        "Conversation confidence: "
        f"{conversation_confidence:.2f}"
    )
    print(f"Policy limitations: {limitations}")
    print(
        "Limitation penalty applied: "
        f"{limitation_penalty_applied}"
    )
    print(
        f"Overall confidence: {final_confidence:.2f}"
    )

    return final_confidence

In [83]:
from datetime import datetime, timezone


def check_escalation_needed(confidence_score):
    """
    Determine whether the generated response should be escalated for human review.
    """
    if confidence_score < CONFIDENCE_THRESHOLD:
        return (True, f"Low Confidence ({confidence_score:.2f})")

    return (False, f"OK ({confidence_score:.2f})")


def create_escalation_packet(question, confidence):
    """
    Create an escalation record for a low-confidence response.
    """
    return {
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),
        "question": question,
        "confidence_score": confidence,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "reason": "Confidence below threshold",
        "require_human_review": True
    }

In [89]:
import json
import time


# Stores latency measurements across multiple requests
latency_metrics = {
    "parallel_agents": [],
    "reasoning_aggregator": [],
    "total_pipeline": []
}


async def agentic_rag_answer_with_tracking(
    question,
    guest_type,
    loyalty,
    city,
    session_id,
    use_cache=True
):
    """
    Run the complete Aurora Agentic RAG workflow.

    The function:

    1. Retrieves recent conversation memory.
    2. Determines whether the question is suitable for caching.
    3. Checks the persistent response cache.
    4. Runs the Policy and Conversation Agents in parallel.
    5. Calculates the combined confidence score.
    6. Checks whether human escalation is required.
    7. Calls the Reasoning Aggregator.
    8. Stores the interaction in conversation memory.
    9. Stores the generated response in the persistent cache.
    10. Tracks execution latency.
    """

    # Start the total pipeline timer.
    total_start_time = time.time()

   
    # Retrieve recent conversation memory
    chat_history_tuples = get_chat_history_tuples(
        session_id=session_id,
        limit=6
    )

    chat_history_text = get_chat_history_text(
        session_id=session_id,
        limit=6
    )

 
    # Decide whether this question should use the cache
    should_use_cache = (
        use_cache
        and is_cacheable_question(question)
    )

    # Check the persistent response cache
    if should_use_cache:
        cached_response = get_cached_response(
            question=question,
            guest_type=guest_type,
            loyalty=loyalty,
            city=city,
            session_id=session_id
        )

        if cached_response:
            cached_response["cache_hit"] = True

            # Measure the latency of this cache retrieval,
            # rather than returning the original generation time.
            cached_response["latency_ms"] = (
                time.time() - total_start_time
            ) * 1000

            answer_text = (
                cached_response.get("text")
                or cached_response.get("answer")
                or cached_response.get("result")
            )

            if not answer_text:
                raise ValueError(
                    "The cached response does not contain "
                    "a recognised answer field."
                )

            # A cache-hit response is still part of the current
            # conversation and should be stored in memory.
            store_conversation(
                session_id,
                "user",
                question
            )

            store_conversation(
                session_id,
                "assistant",
                answer_text
            )

            print(
                "Cache hit: previously stored response returned."
            )

            return cached_response

    # Run both specialist agents in parallel
    parallel_agents_start = time.time()

    policy_output, conversation_output = (
        await run_agents_parallel(
            question=question,
            guest_type=guest_type,
            loyalty=loyalty,
            city=city,
            chat_history_tuples=chat_history_tuples,
            chat_history_text=chat_history_text
        )
    )

    parallel_agents_latency = (
        time.time() - parallel_agents_start
    ) * 1000


    # Calculate the combined confidence score
    confidence = calculate_confidence_score(
        policy_output=policy_output,
        conversation_output=conversation_output
    )

    # Check whether human escalation is required
    needs_escalation, escalation_reason = (
        check_escalation_needed(confidence)
    )

 
    # Run the Reasoning Aggregator
    reasoning_aggregator_start = time.time()

    final_answer = aggregator_chain.invoke(
        {
            "policy_agent_output": json.dumps(
                policy_output,
                ensure_ascii=False,
                default=str
            ),
            "conversation_agent_output": json.dumps(
                conversation_output,
                ensure_ascii=False,
                default=str
            ),
            "chat_history_text": chat_history_text,
            "question": question
        }
    )

    reasoning_aggregator_latency = (
        time.time() - reasoning_aggregator_start
    ) * 1000

  
    # Safely extract the final answer
    if isinstance(final_answer, dict):
        answer_text = (
            final_answer.get("text")
            or final_answer.get("answer")
            or final_answer.get("result")
        )
    else:
        answer_text = str(final_answer)

    if not answer_text:
        raise ValueError(
            "No recognised answer field was returned by "
            f"aggregator_chain: {final_answer}"
        )

  
    # Create an escalation packet where necessary
    escalation_packet = None

    if needs_escalation:
        print(
            f"Escalation needed: {escalation_reason}"
        )

        escalation_packet = create_escalation_packet(
            question=question,
            confidence=confidence
        )

    else:
        print(
            f"Confidence score: {confidence:.2f}"
        )

    # Save the interaction to conversation memory
    store_conversation(
        session_id,
        "user",
        question)

    store_conversation(
        session_id,
        "assistant",
        answer_text)

    # Calculate and store latency measurements
    total_pipeline_latency = (
        time.time() - total_start_time
    ) * 1000

    latency_metrics["parallel_agents"].append(
        parallel_agents_latency
    )

    latency_metrics["reasoning_aggregator"].append(
        reasoning_aggregator_latency
    )

    latency_metrics["total_pipeline"].append(
        total_pipeline_latency
    )

   
    # Create the complete response object
    response = {
        "text": answer_text,
        "confidence": confidence,
        "needs_escalation": needs_escalation,
        "escalation_reason": (
            escalation_reason
            if needs_escalation
            else None
        ),
        "escalation_packet": escalation_packet,
        "latency_ms": total_pipeline_latency,
        "parallel_agents_latency_ms": (
            parallel_agents_latency
        ),
        "reasoning_aggregator_latency_ms": (
            reasoning_aggregator_latency
        ),
        "policy_output": policy_output,
        "conversation_output": conversation_output,
        "cache_hit": False
    }


    # Store the newly generated response in the cache
    if should_use_cache:
        cache_response(
            question=question,
            guest_type=guest_type,
            loyalty=loyalty,
            city=city,
            response=response,
            session_id=session_id
        )

        print(
            "Response successfully stored in the "
            "persistent cache."
        )

    elif use_cache:
        print(
            "Question was identified as context-dependent, "
            "so the response was not cached."
        )

    return response

In [85]:
def show_metrics_dashboard():
    """
    Display operational metrics for the Aurora Agentic RAG
    system, including cache, latency, cost and configuration.
    """

    print("=" * 55)
    print("Aurora Agentic RAG Performance Dashboard")
    print("=" * 55)

  
    # Cache metrics
    print("\nCache Metrics")

    cache_summary = get_cache_summary(display=False)

    print(
        f"Cached responses: "
        f"{cache_summary['stored_entries']}"
    )

    print(
        f"Cache checks: "
        f"{cache_summary['total_requests']}"
    )

    print(
        f"Cache hit rate: "
        f"{cache_summary['hit_rate']:.1%} "
        f"({cache_summary['hits']} hits, "
        f"{cache_summary['misses']} misses)"
    )

    print(
        f"Total recorded cache accesses: "
        f"{cache_summary['total_cache_accesses']}"
    )

    if cache_summary["last_accessed_at"]:
        print(
            f"Last cache access: "
            f"{cache_summary['last_accessed_at']}"
        )

  
    # Latency metrics
    print("\nLatency Metrics")

    latency_data_available = False

    for component, latencies in latency_metrics.items():
        if latencies:
            latency_data_available = True

            average_latency = (
                sum(latencies) / len(latencies)
            )

            minimum_latency = min(latencies)
            maximum_latency = max(latencies)

            readable_name = (
                component.replace("_", " ").title()
            )

            print(
                f"{readable_name}: "
                f"{average_latency:.0f} ms average "
                f"({minimum_latency:.0f}–"
                f"{maximum_latency:.0f} ms, "
                f"{len(latencies)} requests)"
            )

    if not latency_data_available:
        print("No latency measurements recorded yet.")

   
    # Cost metrics
    print("\nCost Metrics")

    if cost_log:
        total_cost = sum(cost_log)

        print(
            f"Tracked model calls: "
            f"{len(token_usage_log)}"
        )

        print(
            f"Total estimated cost: "
            f"${total_cost:.6f}"
        )

        print(
            f"Average estimated cost per call: "
            f"${total_cost / len(cost_log):.6f}"
        )

    else:
        print("No token usage or cost data recorded yet.")

 
    # System configuration
    print("\nSystem Configuration")

    print(
        f"Confidence threshold: "
        f"{CONFIDENCE_THRESHOLD:.2f}"
    )

    print(
        f"Maximum cache entries: "
        f"{MAX_CACHE_ENTRIES}"
    )

    print(
        f"Persistent cache database: "
        f"{CACHE_DB_PATH}"
    )

    print("=" * 55)

In [90]:
# TEST THE CACHE MECHANISM

from uuid import uuid4


async def test_agents():
    """
    Test the Aurora Agentic RAG pipeline and persistent cache.

    The first request should produce a cache miss and generate
    a new response. The second identical request should retrieve
    that response from the persistent cache.
    """

    # Use one new session ID for both requests.
    # This prevents an old cache entry from causing Query 1
    # to return a cache hit.
    session_id = f"cache_test_{uuid4().hex}"

    test_request = {
        "question": (
            "Amenities & services "
            "(wifi, breakfast, gym, pool, spa)"
        ),
        "guest_type": "Business",
        "loyalty": "Gold",
        "city": "Sydney",
        "session_id": session_id,
        "use_cache": True
    }

   
    # First request: expected cache miss
    print("\n" + "=" * 50)
    print("QUERY 1 — EXPECTED CACHE MISS")
    print("=" * 50)

    response1 = await agentic_rag_answer_with_tracking(
        **test_request
    )

    response_text1 = response1.get(
        "text",
        "No response returned"
    )

    confidence1 = response1.get("confidence")
    cache_hit1 = response1.get("cache_hit")

    print(f"\nResponse 1: {response_text1}")

    if confidence1 is not None:
        print(f"Confidence 1: {confidence1:.2f}")
    else:
        print("Confidence 1: not returned")

    if cache_hit1 is not None:
        print(f"Cache hit 1: {cache_hit1}")
    else:
        print("Cache status 1: not returned")

   
    # Second request: expected cache hit
    print("\n" + "=" * 50)
    print("QUERY 2 — EXPECTED CACHE HIT")
    print("=" * 50)

    response2 = await agentic_rag_answer_with_tracking(
        **test_request
    )

    response_text2 = response2.get(
        "text",
        "No response returned"
    )

    confidence2 = response2.get("confidence")
    cache_hit2 = response2.get("cache_hit")

    print(f"\nResponse 2: {response_text2}")

    if confidence2 is not None:
        print(f"Confidence 2: {confidence2:.2f}")
    else:
        print("Confidence 2: not returned")

    if cache_hit2 is not None:
        print(f"Cache hit 2: {cache_hit2}")
    else:
        print("Cache status 2: not returned")

   
    # Compare the responses
    print("\n" + "=" * 50)
    print("CACHE TEST RESULTS")
    print("=" * 50)

    responses_identical = (
        response_text1 == response_text2
    )

    cache_test_passed = (
        cache_hit1 is False
        and cache_hit2 is True
    )

    if responses_identical:
        print("Both responses are identical.")
    else:
        print("The responses are different.")

    if cache_test_passed:
        print(
            "Cache test passed: the first request was a miss "
            "and the second request was a hit."
        )

    elif cache_hit2 is True:
        print(
            "The second response came from the cache, but the "
            "first request was not confirmed as a cache miss."
        )

    else:
        print(
            "The cache test could not be confirmed. Check the "
            "cache lookup and cache storage functions."
        )

  
    # Compare request latency
    first_latency = response1.get("latency_ms")
    second_latency = response2.get("latency_ms")

    if (
        first_latency is not None
        and second_latency is not None
    ):
        print(
            f"First-request latency: "
            f"{first_latency:.0f} ms"
        )

        print(
            f"Second-request latency: "
            f"{second_latency:.0f} ms"
        )

        if (
            first_latency > 0
            and second_latency < first_latency
        ):
            latency_reduction = (
                (first_latency - second_latency)
                / first_latency
                * 100
            )

            print(
                f"Cache latency reduction: "
                f"{latency_reduction:.1f}%"
            )

    # Display system metrics after both requests.
    show_metrics_dashboard()

    return {
        "session_id": session_id,
        "response1": response1,
        "response2": response2,
        "responses_identical": responses_identical,
        "cache_test_passed": cache_test_passed
    }


responses = await test_agents()


QUERY 1 — EXPECTED CACHE MISS
Cache miss: no matching response found.
Confidence calculation
Policy confidence: 0.60
Conversation confidence: 0.80
Policy limitations: ['No information on wifi, breakfast, gym, or pool amenities was found.']
Limitation penalty applied: True
Overall confidence: 0.54
Escalation needed: Low Confidence (0.54)
Response successfully stored in the persistent cache.

Response 1: Thank you for your inquiry about our amenities and services. While I can confirm that we have multiple restaurants on property and offer poolside service from 11:00 AM to 6:00 PM, I currently do not have specific information regarding Wi-Fi, breakfast, gym, or pool amenities. For breakfast, it typically depends on your rate and package, and I recommend checking with our customer support for the most accurate details. Additionally, our spa services are available daily from 8:00 AM to 8:00 PM, and advance reservations are recommended. If you have any further questions or need assistance, 

In [92]:
# TEST QUERIES FOR AGENTIC RAG EVALUATION

test_queries = [

    # Test 1: General amenities enquiry
    {
        "question": (
            "What amenities and services are available "
            "(Wi-Fi, breakfast, gym, pool and spa)?"
        ),
        "guest_type": "Business",
        "loyalty": "Gold",
        "city": "Sydney"
    },

    # Test 2: Pet policy
    {
        "question": (
            "I will be travelling with my pet. "
            "Is the property pet-friendly?"
        ),
        "guest_type": "Leisure",
        "loyalty": "Gold",
        "city": "Toronto"
    },

    # Test 3: Breakfast policy
    {
        "question": (
            "Is breakfast included, and how is it served?"
        ),
        "guest_type": "Business",
        "loyalty": "Silver",
        "city": "London"
    },

    # Test 4: Repeat of Test 2
    # Used to verify that the persistent cache returns a cache hit for an identical request.
    {
        "question": (
            "I will be travelling with my pet. "
            "Is the property pet-friendly?"
        ),
        "guest_type": "Leisure",
        "loyalty": "Gold",
        "city": "Toronto"
    }
]

In [93]:
# BATCH TEST OF THE AGENTIC RAG PIPELINE
import json
from uuid import uuid4


def normalise_evaluation_score(score, default=0.0):
    """
    Convert an evaluation result into a numeric score.

    This handles numeric values, numeric strings and dictionaries
    containing a recognised score field.
    """

    if isinstance(score, dict):
        score = (
            score.get("score")
            or score.get("faithfulness")
            or score.get("relevance")
            or default
        )

    try:
        return float(score)

    except (TypeError, ValueError):
        return float(default)


async def test_agents_again():
    """
    Test the Aurora Agentic RAG system using multiple requests.

    The function records:

    - generated response
    - confidence score
    - faithfulness score
    - relevance score
    - execution latency
    - cache-hit status
    """

    results = []

    # Generate one new session ID for this complete test run.
    # All requests within the test use the same session.
    session_id = f"batch_test_{uuid4().hex}"

    # Keep track of request signatures already encountered.
    seen_requests = set()

    for i, test in enumerate(test_queries, start=1):

        # Create the same logical signature used by the cache.
        request_signature = (
            test["question"].strip().lower(),
            test["guest_type"].strip().lower(),
            test["loyalty"].strip().lower(),
            test["city"].strip().lower(),
            session_id.lower()
        )

        expected_cache_hit = (
            request_signature in seen_requests
        )

        print("\n" + "=" * 60)
        print(f"QUERY {i}")
        print("=" * 60)

        print(f"Question: {test['question']}")
        print(f"Guest type: {test['guest_type']}")
        print(f"Loyalty: {test['loyalty']}")
        print(f"City: {test['city']}")
        print(
            f"Expected cache hit: "
            f"{expected_cache_hit}"
        )

      
        # Run the complete Agentic RAG workflow
        response = await agentic_rag_answer_with_tracking(
            question=test["question"],
            guest_type=test["guest_type"],
            loyalty=test["loyalty"],
            city=test["city"],
            session_id=session_id,
            use_cache=True
        )

        # Record the signature after the request has run.
        seen_requests.add(request_signature)

      
        # Extract response information
        answer_text = response.get(
            "text",
            "No response returned"
        )

        confidence = response.get("confidence")

        latency_ms = response.get(
            "latency_ms",
            0.0
        )

        cache_hit = response.get(
            "cache_hit",
            False
        )

        print(f"\nResponse: {answer_text}")

        if confidence is not None:
            print(
                f"Confidence: "
                f"{confidence:.2f}"
            )
        else:
            print("Confidence: not returned")

        print(f"Cache hit: {cache_hit}")
        print(f"Latency: {latency_ms:.2f} ms")

        # Prepare retrieved evidence for evaluation  
        policy_output = response.get(
            "policy_output",
            {}
        )

        conversation_output = response.get(
            "conversation_output",
            {}
        )

        evaluation_context = (
            "POLICY AGENT OUTPUT:\n"
            f"{json.dumps(policy_output, indent=2, default=str)}"
            "\n\n"
            "CONVERSATION AGENT OUTPUT:\n"
            f"{json.dumps(conversation_output, indent=2, default=str)}"
        )

     
        # Evaluate response quality
        faithfulness_result = evaluate_faithfulness(
            answer_text,
            evaluation_context
        )

        relevance_result = evaluate_relevance(
            answer_text,
            test["question"]
        )

        faithfulness = normalise_evaluation_score(
            faithfulness_result
        )

        relevance = normalise_evaluation_score(
            relevance_result
        )

        print(
            f"Faithfulness: "
            f"{faithfulness:.2f}"
        )

        print(
            f"Relevance: "
            f"{relevance:.2f}"
        )

        cache_behaviour_correct = (
            cache_hit == expected_cache_hit
        )

        print(
            f"Expected cache behaviour confirmed: "
            f"{cache_behaviour_correct}"
        )

     
        # Store the test result
        result = {
            "query_number": i,
            "session_id": session_id,
            "question": test["question"],
            "guest_type": test["guest_type"],
            "loyalty": test["loyalty"],
            "city": test["city"],
            "response": answer_text,
            "confidence": confidence,
            "faithfulness": faithfulness,
            "relevance": relevance,
            "latency_ms": latency_ms,
            "expected_cache_hit": expected_cache_hit,
            "cache_hit": cache_hit,
            "cache_behaviour_correct": (
                cache_behaviour_correct
            )
        }

        results.append(result)

    # Display final test summary
    print("\n" + "=" * 60)
    print("BATCH TEST SUMMARY")
    print("=" * 60)

    total_queries = len(results)

    cache_hits = sum(
        1
        for result in results
        if result["cache_hit"]
    )

    correct_cache_results = sum(
        1
        for result in results
        if result["cache_behaviour_correct"]
    )

    valid_confidences = [
        result["confidence"]
        for result in results
        if result["confidence"] is not None
    ]

    average_confidence = (
        sum(valid_confidences) / len(valid_confidences)
        if valid_confidences
        else 0.0
    )

    average_faithfulness = (
        sum(
            result["faithfulness"]
            for result in results
        ) / total_queries
        if total_queries
        else 0.0
    )

    average_relevance = (
        sum(
            result["relevance"]
            for result in results
        ) / total_queries
        if total_queries
        else 0.0
    )

    average_latency = (
        sum(
            result["latency_ms"]
            for result in results
        ) / total_queries
        if total_queries
        else 0.0
    )

    print(f"Total queries: {total_queries}")
    print(f"Cache hits: {cache_hits}")
    print(
        f"Correct cache results: "
        f"{correct_cache_results}/{total_queries}"
    )
    print(
        f"Average confidence: "
        f"{average_confidence:.2f}"
    )
    print(
        f"Average faithfulness: "
        f"{average_faithfulness:.2f}"
    )
    print(
        f"Average relevance: "
        f"{average_relevance:.2f}"
    )
    print(
        f"Average latency: "
        f"{average_latency:.2f} ms"
    )

    show_metrics_dashboard()

    return results


results = await test_agents_again()


# Display each stored result
for result in results:
    print("\n", result)


QUERY 1
Question: What amenities and services are available (Wi-Fi, breakfast, gym, pool and spa)?
Guest type: Business
Loyalty: Gold
City: Sydney
Expected cache hit: False
Cache miss: no matching response found.
Confidence calculation
Policy confidence: 0.60
Conversation confidence: 0.80
Policy limitations: ['Wi-Fi, breakfast, gym, and pool information is not available.']
Limitation penalty applied: True
Overall confidence: 0.54
Escalation needed: Low Confidence (0.54)
Response successfully stored in the persistent cache.

Response: Thank you for your inquiry about our amenities and services. We offer multiple restaurants on property, room service from 7:00 AM to 11:00 PM, and spa services available daily from 8:00 AM to 8:00 PM, with advance reservations recommended and a 24-hour cancellation policy. Unfortunately, I don't have specific information regarding Wi-Fi, breakfast, gym, and pool availability at this time. For the most accurate details, especially regarding breakfast inclu